In [172]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Shape:", df.shape)
print()
print(df["split"].value_counts())

Shape: (12515, 24)

split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [173]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

In [174]:
spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

In [175]:
all_vital_features = spo2_hr_features + [
    "respiratory_rate",
    "rr_lag_1",
    "rr_rolling_mean_5",
    "mean_arterial_pressure",
    "map_lag_1",
    "map_rolling_mean_5"
]

target = "future_spo2_5min"

In [176]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

In [177]:
common_val_df = val_df.dropna(
    subset=[
        "spo2",
        target
    ]
).copy()

print(
    "Common validation rows:",
    len(common_val_df)
)

Common validation rows: 1748


In [178]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [179]:
def evaluate_ridge(
    feature_list,
    feature_name,
    alpha=1.0
):

    train_ready = train_df.dropna(
        subset=[target]
    ).copy()

    X_train = train_ready[feature_list]
    y_train = train_ready[target]

    X_val = common_val_df[feature_list]
    y_val = common_val_df[target]

    model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(
                    alpha=alpha
                )
            )
        ]
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_val
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            predictions
        )
    )

    r2 = r2_score(
        y_val,
        predictions
    )

    result = {
        "Model": feature_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    return model, result

In [180]:
ridge_spo2_model, ridge_spo2_result = evaluate_ridge(
    spo2_features,
    "Ridge - SpO2 History",
    alpha=1.0
)

ridge_spo2_result

{'Model': 'Ridge - SpO2 History',
 'MAE': 0.4204565487923267,
 'RMSE': np.float64(1.3628500050160381),
 'R2': 0.11874507722145566}

In [181]:
ridge_hr_model, ridge_hr_result = evaluate_ridge(
    spo2_hr_features,
    "Ridge - SpO2 + HR",
    alpha=1.0
)

ridge_hr_result

{'Model': 'Ridge - SpO2 + HR',
 'MAE': 0.4253690834650425,
 'RMSE': np.float64(1.3635047548677763),
 'R2': 0.11789811661471661}

In [182]:
ridge_all_model, ridge_all_result = evaluate_ridge(
    all_vital_features,
    "Ridge - All Vitals",
    alpha=1.0
)

ridge_all_result

{'Model': 'Ridge - All Vitals',
 'MAE': 0.5124472673292619,
 'RMSE': np.float64(1.3844666927380442),
 'R2': 0.09056752195555451}

In [183]:
results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

previous_results = pd.read_csv(
    results_file
)

previous_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610


In [184]:
ridge_results = pd.DataFrame(
    [
        ridge_spo2_result,
        ridge_hr_result,
        ridge_all_result
    ]
)

In [185]:
all_results = pd.concat(
    [
        previous_results,
        ridge_results
    ],
    ignore_index=True
)

all_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568


In [186]:
all_results.sort_values(
    "MAE"
)

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568


In [187]:
all_results.sort_values(
    "MAE"
)

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568


In [188]:
baseline_mae = all_results.loc[
    all_results["Model"]
    == "Persistence Baseline",
    "MAE"
].iloc[0]

In [189]:
ridge_results["MAE_Improvement_vs_Baseline_%"] = (
    (
        baseline_mae
        - ridge_results["MAE"]
    )
    /
    baseline_mae
) * 100

ridge_results

,Model,MAE,RMSE,R2,MAE_Improvement_vs_Baseline_%
0,Ridge - SpO2 History,0.420457,1.362850,0.118745,-57.547277
1,Ridge - SpO2 + HR,0.425369,1.363505,0.117898,-59.388030
2,Ridge - All Vitals,0.512447,1.384467,0.090568,-92.016682


In [190]:
all_results.to_csv(
    results_file,
    index=False
)

print(
    "Updated model comparison saved!"
)

Updated model comparison saved!
